# AI Exercises – Heuristic Search

<i>
Jan Schlüter, Tara Jadidi, Tobias Morocutti, Patricia Hu, Alexander Fichtinger, Nikita Lvov
<br/>
Institute of Computational Perception, Johannes Kepler University Linz
</i>

In this notebook, you will implement **heuristic search** algorithms from the lecture, Greedy Best First Search and A\*, and apply them to two problems: the **journey planner** from last time, and **sliding-tile puzzles**. You will see how much a good heuristic saves, what makes one heuristic better than another, and what happens if a heuristic is *not* admissible.

## If needed: Google Colab Setup

**Only if you're running this notebook on Google Colab**, run the next cell to fetch the other files it depends on (helper modules, maps).

**If you're running locally or on Binder**, this is not needed, and it will do nothing. Just skip it.

In [ ]:
# If you're running this cell on Google Colab, it will copy in the remaining files.
# Otherwise, it doesn't do anything.

import os, sys

if 'google.colab' in sys.modules and not os.path.exists('puzzle.py'):
    if not os.path.exists('ai-exercises'):
        !git clone --quiet --branch topic02 --depth 1 https://github.com/CPJKU/ai-exercises.git
    %cd ai-exercises

## Recap: Search algorithms and the framework

We use the same framework as last time: a `Problem` offers `get_start()`, `is_goal(state)` and `successors(state)`, and search algorithms wrap states into `Node`s that remember their parent, action, depth, and (cumulative!) path cost.

Two small additions for this notebook, so we can compare algorithms:
- `expand()` now counts how many nodes were expanded, and optionally logs them.
- `measure()` runs an algorithm on a problem and returns the solution along with the number of expansions. It can give up after a budget of expansions, so we can test algorithms that would otherwise run for hours.

In [ ]:
import random
import time
from functools import partial
from typing import Callable, NamedTuple

from priority_queue import PriorityQueue
from journey import (load_map, print_journey, straight_line_km, COSTS,
                     JourneyState, JourneyProblem, TransitMap)
# zoomable versions of the map drawings from journey.py (scroll to zoom, drag to pan)
from zoom import draw_journey_zoom, draw_journey_expansions_zoom
import importlib, zoom
importlib.reload(zoom)
from zoom import draw_journey_expansions_zoom

In [ ]:
class State:
    """A world state (here: a stop in a transit network, or a board of a sliding puzzle)."""

class Action:
    """An action in a world (here: a ride, or a move of a tile)."""

class Problem:
    """The interface of all search problems, as in the last notebook."""
    def get_start(self) -> State: ...
    def is_goal(self, state: State) -> bool: ...
    def successors(self, state: State) -> list[tuple[Action, State]]: ...

# A heuristic estimates the remaining cost from a state to the goal of a problem
Heuristic = Callable[[State, Problem], float]


class Node:
    """
    A step in a plan, pointing at a world state and possibly at a preceding step and action.
    """
    def __init__(self, state: State, parent: "Node | None" = None, action: Action | None = None):
        self.state = state
        if (parent is None) ^ (action is None):
            raise ValueError("parent and action must be given or omitted together")
        self.parent = parent
        self.action = action
        self.depth = 0 if parent is None else 1 + parent.depth
        self.cost = 0 if parent is None else getattr(action, 'cost', 0) + parent.cost


class BudgetExceeded(Exception):
    pass

logged_expansions = None   # set to a list to log (node, successors) pairs
num_expansions = 0         # number of expansions so far
expansion_budget = None    # if set, abort after this many expansions

def expand(node: Node, problem: Problem) -> list[Node]:
    """
    Wraps all successors of a world state into search graph nodes.
    Counts the expansion, and logs it in case `logged_expansions` is a list.
    """
    global num_expansions
    if expansion_budget is not None and num_expansions >= expansion_budget:
        raise BudgetExceeded()
    num_expansions += 1
    successors = [Node(new_state, parent=node, action=action)
                  for action, new_state in problem.successors(node.state)]
    if isinstance(logged_expansions, list):
        logged_expansions.append((node, successors))
    return successors

In [ ]:
class Result(NamedTuple):
    solution: Node | None
    expansions: int
    seconds: float
    gave_up: bool

def measure(algorithm: Callable[[Problem], Node | None], problem: Problem,
            budget: int | None = None, log: bool = False) -> Result:
    """
    Runs `algorithm(problem)` and returns a Result with the solution node, the
    number of expansions, the time taken, and whether it gave up after
    `budget` expansions (in which case the solution is None). If `log` is
    True, the expansions are available in `logged_expansions` afterwards.
    """
    global num_expansions, expansion_budget, logged_expansions
    num_expansions, expansion_budget = 0, budget
    logged_expansions = [] if log else None
    start, gave_up, solution = time.time(), False, None
    try:
        solution = algorithm(problem)
    except BudgetExceeded:
        gave_up = True
    finally:
        expansion_budget = None
    return Result(solution, num_expansions, time.time() - start, gave_up)

Here is **Uniform Cost Search** from last time, so you do not depend on your own implementation:

In [ ]:
def uniform_cost_search(problem: Problem) -> Node | None:
    closed = set()
    root = Node(problem.get_start())
    fringe = PriorityQueue()
    fringe.push(root, root.cost)
    while fringe:
        node = fringe.pop()
        if problem.is_goal(node.state):
            return node
        if node.state not in closed:
            closed.add(node.state)
            for successor in expand(node, problem):
                fringe.push(successor, successor.cost)
    return None

## First problem: Journey planning, again

We continue with the journey planner on the Vienna network, with all metro, tram and bus lines. We want the route with the shortest **distance** traveled along the tracks. First, let us see how UCS finds a route from Handelskai to Mitte-Landstraße, and which stops it needs to expand to do so:

In [ ]:
vienna_xxl = load_map('vienna', restrict=['metro', 'tram', 'bus'])
problem = vienna_xxl.search_route("Handelskai", "Mitte-Landstraße", cost="distance")

result = measure(uniform_cost_search, problem, log=True)
print(f"UCS: {result.solution.depth} stops, {result.solution.cost:.2f} km, {result.expansions} expansions")
draw_journey_expansions_zoom(logged_expansions, vienna_xxl, result.solution, width=3200, highlight=True, dim=0.6)

UCS is *uninformed*: it only knows how far it has come, so it expands stops in all directions, including those leading away from the goal. We know more than that: we know where the goal is on the map. Let's use it!

### Heuristics

A **heuristic** `h(state, problem)` estimates the remaining cost from a state to the goal. It is **admissible** if it *never overestimates*: for every state, `h` is at most the true cheapest cost to the goal.

For the distance in kilometers, an obvious heuristic is the **straight-line distance** to the goal: no route along tracks can be shorter than the direct line. Stops have a latitude and longitude; `straight_line_km(a, b)` computes the distance between two of them.

Implement the straight-line heuristic for journey problems. The goal state is `problem.goal`.

In [ ]:
def straight_line(state: JourneyState, problem: JourneyProblem) -> float:
    # TODO: return the straight-line distance in km between the state and the problem's goal.
    # Hint: use straight_line_km(), it takes two states.

    pass

To test a heuristic, we can check that it is not larger than the true remaining cost for every state on an optimal solution. (If it is, it is *not* admissible. If it is not, we have not proven anything, but we have not found a counterexample either.)

In [ ]:
def check_admissible(problem: Problem, heuristic: Heuristic) -> None:
    """
    Finds an optimal solution with UCS, and checks the heuristic against the
    true remaining cost for every state along it.
    """
    solution = uniform_cost_search(problem)
    node, checked, violations = solution, 0, []
    while node is not None:
        remaining = solution.cost - node.cost  # what is left on an optimal solution from here
        estimate = heuristic(node.state, problem)
        # small tolerance: sums of floats in different order can differ in the last digits
        if estimate > remaining + 1e-9:
            violations.append((node.state, estimate, remaining))
        checked += 1
        node = node.parent
    if violations:
        for state, estimate, remaining in violations[:5]:
            print(f"{state}: heuristic says {estimate:.2f}, but only {remaining:.2f} remain")
        print(f"NOT admissible: {len(violations)} of {checked} states overestimate the remaining cost")
    else:
        print(f"No overestimates found on the {checked} states of an optimal solution.")

In [ ]:
check_admissible(problem, straight_line)

Feel free to change your heuristic to return a too-small estimate and rerun the admissibility check. Remember to correct the heuristic before you continue, though

### A\* Search

A\* orders the fringe by the cost *so far* plus the *estimated* cost to go: `f(n) = g(n) + h(n)`. Uniform Cost Search is the special case `h = 0`.

Copy your (or our) `uniform_cost_search()` and change only the priority that nodes are pushed to the fringe with. Make the heuristic a parameter. Remember: `node.cost` is the cumulative path cost, the heuristic needs to be called on the *state*.

In [ ]:
def astar_search(problem: Problem, heuristic: Heuristic) -> Node | None:

    # TODO: copy uniform_cost_search() and adapt it:
    # - add the heuristic as a parameter (heuristic(state, problem) -> float)
    # - push nodes to the fringe with priority cost so far + heuristic estimate

    pass

#### Verification

A\* should find the same cheapest route as UCS (13 stops, 5.09 km), but with a fraction of the expansions:

In [ ]:
result = measure(lambda problem: astar_search(problem, straight_line), problem, log=True)
print(f"A*: {result.solution.depth} stops, {result.solution.cost:.2f} km, {result.expansions} expansions")
assert abs(result.solution.cost - 5.09) < 0.01, "A* should find the same cheapest route as UCS"
assert result.expansions < 100, "A* should need far fewer expansions than UCS"
draw_journey_expansions_zoom(logged_expansions, vienna_xxl, result.solution, width=2400)

Instead of expanding stops in all directions, A\* mostly follows the direction to the goal.

### Greedy Best First Search

What if we only look at the estimate, ignoring the cost so far? Greedy Best First Search (GBFS) always expands the node that *looks* closest to the goal.

Implement `greedy_best_first_search()`. Compared to A\*, only one thing changes.

In [ ]:
def greedy_best_first_search(problem: Problem, heuristic: Heuristic) -> Node | None:

    # TODO: copy astar_search() and adapt it:
    # - ignore the cost so far when pushing nodes to the fringe

    pass

In [ ]:
result = measure(lambda problem: greedy_best_first_search(problem, straight_line), problem, log=True)
print(f"GBFS: {result.solution.depth} stops, {result.solution.cost:.2f} km, {result.expansions} expansions")
draw_journey_zoom(result.solution, vienna_xxl, width=1600)

GBFS needs even fewer expansions, but it did not find the cheapest route (5.09 km): It was lured down a route that looked promising at every step. It is *not optimal*.

### Comparing algorithms on many routes

A single route tells us little. To compare algorithms properly, we run them on 100 random routes of at least 5 stops. The helper `compare()` reports the average number of expansions, and how much longer than the cheapest route found by any algorithm the returned routes are.

In [ ]:
def random_routes(transit_map: TransitMap, cost: str, count: int = 100, seed: int = 0,
                  min_stops: int = 5) -> list[JourneyProblem]:
    """
    Returns a list of random journey problems with a solution of at least `min_stops` stops.
    """
    rng = random.Random(seed)
    stops = list(transit_map.stops)
    problems = []
    while len(problems) < count:
        start, goal = rng.sample(stops, 2)
        problem = transit_map.search_route(start, goal, cost=cost)
        solution = uniform_cost_search(problem)
        if solution is not None and solution.depth >= min_stops:
            problems.append(problem)
    return problems


def compare(problems: list[Problem], algorithms: dict[str, Callable[[Problem], Node | None]],
            budget: int | None = None) -> None:
    """
    Runs each algorithm (a function taking a problem) on each problem and
    prints average expansions, and the excess cost over the cheapest solution any
    algorithm found. If an algorithm exceeds the expansion budget, it gives up.
    """
    results = {name: [measure(algorithm, problem, budget=budget) for problem in problems]
               for name, algorithm in algorithms.items()}
    best = [min(runs[i].solution.cost for runs in results.values() if runs[i].solution is not None)
            for i in range(len(problems))]
    print(f"{'algorithm':<36}{'expansions':>11}{'seconds':>9}{'avg excess cost':>17}{'not optimal':>13}{'gave up':>9}")
    for name, runs in results.items():
        solved = [(run, best[i]) for i, run in enumerate(runs) if run.solution is not None]
        if not solved:
            print(f"{name:<36}{'-':>11}{sum(run.seconds for run in runs):>9.1f}{'-':>17}{'-':>13}{len(runs):>9}")
            continue
        expansions = sum(run.expansions for run, _ in solved) / len(solved)
        excess = [100 * (run.solution.cost / opt - 1) for run, opt in solved]
        wrong = sum(e > 1e-6 for e in excess)
        gave_up = len(runs) - len(solved)
        print(f"{name:<36}{expansions:>11.0f}{sum(run.seconds for run in runs):>9.1f}"
              f"{sum(excess) / len(excess):>16.2f}%{f'{wrong}/{len(solved)}':>13}{gave_up:>9}")

In [ ]:
routes = random_routes(vienna_xxl, 'distance')
compare(routes, {
    'Uniform Cost Search': uniform_cost_search,
    'A* (straight-line)': lambda p: astar_search(p, straight_line),
    'GBFS (straight-line)': lambda p: greedy_best_first_search(p, straight_line),
})

A\* with an admissible heuristic still finds an optimal route every time, with a fraction of UCS' expansions. GBFS expands even less, but returns longer routes in most cases.

**Remember:** A\* is only guaranteed to be optimal if the heuristic is admissible. (For the graph search version we use, with the closed set, it must also be *consistent*; see the optional detour below.)

### Optional detour: Inconsistent heuristics

Note that in contrast to the lecture, we used a closed set in our implementation of A\*. To ensure it finds an optimal path, this requires the heuristic to not only be admissible, but also **consistent**:

$$h(s) \le \mathrm{cost}(a) + h(s')$$

for every state $s$ and every action $a$ leading from $s$ to a successor state $s'$. In words: the estimate may not drop by more than the cost of the step. (Every consistent heuristic is admissible, but not the other way round.)

An example for an inconsistent, but still admissible heuristic would be a mix of two heuristics: Straight-line distance for metro stops, zero distance for all others. At a metro stop it estimates the remaining distance, but after riding to a stop without a metro line, the estimate suddenly drops to zero, by much more than the ride costs.

Here it is, already implemented:

In [ ]:
metro_stops = set(load_map('vienna', restrict=['metro']).stops)

def inconsistent(state: JourneyState, problem: JourneyProblem) -> float:
    """
    Straight-line distance for metro stops, zero for all others. Never larger than
    the straight-line distance, so admissible, but not consistent.
    """
    return straight_line(state, problem) if state.stop in metro_stops else 0.0

It is still admissible, as the check confirms. But how does A\* with a closed set do with it on our 100 routes?

In [ ]:
check_admissible(problem, inconsistent)
compare(routes, {
    'A* (straight-line)': lambda p: astar_search(p, straight_line),
    'A* (inconsistent)': lambda p: astar_search(p, inconsistent),
})

Even though it never overestimates, A\* with the inconsistent heuristic expands more stops than with the straight-line distance (it knows less), and in more than half of the routes it returns a route that is longer than necessary: it closed some stops before it had found the cheapest way to them.

### Optional detour: A better admissible heuristic with landmarks

Is the straight-line distance the best admissible heuristic we can get? Of two admissible heuristics, the one that returns *larger* values is better, since it is closer to the true cost. We say it **dominates** the other: A\* with it never expands more nodes (up to tie-breaking). The extreme cases are `h = 0` (UCS) and `h` = the true remaining cost (A\* walks straight to the goal).

Here is a trick to obtain a tighter one. First, some notation: $d(a, b)$ is the **cheapest total cost of any route from stop $a$ to stop $b$** (in km or minutes, like our cost). It is what UCS finds when searching from $a$ to $b$. Routes are directed, so $d(a, b)$ and $d(b, a)$ can differ. The true remaining cost from a stop $s$ to the goal $g$ is $d(s, g)$: exactly what a perfect heuristic would return, but we do not know it during the search.

Now pick a few **landmark** stops $L$, and compute the true cheapest costs $d(L, x)$ and $d(x, L)$ for every stop $x$ once, before searching (using a Uniform Cost Search that never stops, known as Dijkstra's algorithm). Then for any stop $s$ and goal $g$, the triangle inequality (a detour via $s$ cannot be cheaper than the cheapest route) tells us:
- $d(L, g) \le d(L, s) + d(s, g)$, so $d(s, g) \ge d(L, g) - d(L, s)$
- $d(s, L) \le d(s, g) + d(g, L)$, so $d(s, g) \ge d(s, L) - d(g, L)$
Both right-hand sides are lower bounds on the true remaining cost $d(s, g)$, and only use distances we computed in advance. Taking the largest of all these lower bounds, and of the straight-line distance, gives a heuristic that is still admissible, but tighter.

<img src="img/landmark_problem.png" width="800">

Too optimistic! Straight-line distance ignores physical barriers like rivers, bridges, and winding track routes, yielding a much lower estimate (1 km) than the true shortest path (4 km) and causing A\* to expand many unnecessary nodes.

<img src="img/landmark_problem_2.png" width="800">

In [ ]:
Landmark = tuple[dict[str, float], dict[str, float]]  # distances from and to a landmark: d(L, x) and d(x, L)

def distances_via(transit_map: TransitMap, source: str, cost: str,
                  reverse: bool = False) -> dict[str, float]:
    """
    Returns the cheapest cost d(source, x) from `source` to every stop x
    (as a dict: stop name -> cost), or, with reverse=True, the cheapest cost
    d(x, source) from every stop x to `source`.

    This is Uniform Cost Search that does not stop at a goal: it keeps going
    until it has reached every stop it can.
    """
    # Step 1: list the rides leaving each stop, as (next stop, cost of the ride).
    # For the reverse direction, we turn every ride around: a ride a -> b becomes
    # b -> a. Searching outwards from `source` over the turned-around rides finds
    # the cheapest ways *to* `source` on the original map.
    adjacency = {}
    for origin, connections in transit_map.connections.items():
        for connection in connections:
            a, b = (connection.stop, origin) if reverse else (origin, connection.stop)
            adjacency.setdefault(a, []).append((b, COSTS[cost](connection)))

    # Step 2: spread outwards from the source, always continuing from the stop
    # with the lowest cost so far.
    distance = {source: 0.0}             # cheapest cost found so far for each reached stop
    fringe = PriorityQueue()             # stops still to continue from, cheapest first
    fringe.push(source, 0.0)
    while fringe:
        stop = fringe.pop()
        for neighbor, step_cost in adjacency.get(stop, []):
            # Is going to `neighbor` via `stop` cheaper than any way found so far
            # (or is it the first way we found)? If so, remember it, and continue from
            # `neighbor` later. (A stop can end up in the fringe more than once if we
            # find a cheaper way after the first; that is harmless, because we always
            # look up its latest, cheapest cost in `distance`.)
            if distance[stop] + step_cost < distance.get(neighbor, float('inf')):
                distance[neighbor] = distance[stop] + step_cost
                fringe.push(neighbor, distance[neighbor])
    return distance


def choose_landmarks(transit_map: TransitMap, cost: str, count: int = 4) -> list[Landmark]:
    """
    Picks stops that are far from each other, and returns for each the
    distances from it and to it.
    """
    names = list(transit_map.stops)
    landmarks = [max(names, key=lambda n: straight_line_km(transit_map.stops[names[0]], transit_map.stops[n]))]
    while len(landmarks) < count:
        landmarks.append(max(names, key=lambda n: min(straight_line_km(transit_map.stops[n], transit_map.stops[l])
                                                     for l in landmarks)))
    return [(distances_via(transit_map, l, cost), distances_via(transit_map, l, cost, reverse=True))
            for l in landmarks]

Complete the function that computes the landmark estimate for a state. Here, `from_landmark[name]` is $d(L, \text{name})$, the cheapest cost from the landmark to a stop, and `to_landmark[name]` is $d(\text{name}, L)$, the cheapest cost from a stop to the landmark. (Some stops may not be reachable from a landmark, so check for membership first.)

Note that the function takes the `landmarks` and a `base` heuristic as extra parameters. A search algorithm only calls `heuristic(state, problem)`, so we will use [`functools.partial`](https://docs.python.org/3/library/functools.html#functools.partial) to fill in the extra parameters in advance: `partial(landmark_heuristic, landmarks=..., base=...)` is a new function that only needs `state` and `problem`.

In [ ]:
def landmark_heuristic(state: JourneyState, problem: JourneyProblem,
                       landmarks: list[Landmark], base: Heuristic) -> float:
    stop, goal = state.stop, problem.goal.stop
    estimate = base(state, problem)
    for from_landmark, to_landmark in landmarks:
        # TODO: raise the estimate to the larger of the two landmark lower bounds,
        # if they are larger than the current estimate:
        # - from_landmark[goal] - from_landmark[stop]
        # - to_landmark[stop] - to_landmark[goal]
        pass
    return estimate

In [ ]:
distance_landmarks = choose_landmarks(vienna_xxl, 'distance')
straight_line_landmarks = partial(landmark_heuristic, landmarks=distance_landmarks, base=straight_line)
check_admissible(problem, straight_line_landmarks)

In [ ]:
compare(routes, {
    'A* (straight-line)': lambda p: astar_search(p, straight_line),
    'A* (straight-line + 4 landmarks)': lambda p: astar_search(p, straight_line_landmarks),
})

The tighter heuristic expands fewer nodes, and A\* still finds optimal routes. (The price: some precomputation before the first search. It pays off when answering many queries on the same map.)

### What if the heuristic is not admissible?

A\* is only optimal for admissible heuristics. What if we simply *overestimate*? Let's multiply the heuristic by a weight `w`. For `w = 1` nothing changes, for `w > 1` the heuristic may now be larger than the true cost: it is *not admissible*. (This is known as *weighted A\**.)

Implement a function that returns the weight times the estimate of another heuristic. Just as for the landmarks, we use `functools.partial` to fill in the extra parameters in advance: `partial(weighted_heuristic, heuristic=straight_line, weight=2)` is a heuristic that only needs `state` and `problem`.

In [ ]:
def weighted_heuristic(state: State, problem: Problem, heuristic: Heuristic, weight: float) -> float:

    # TODO: return the weight times the heuristic's estimate for this state

    pass

Let's try it on a longer route, from Laudongasse to Nästlbergergasse:

In [ ]:
long_route = vienna_xxl.search_route("Laudongasse", "Nästlbergergasse", cost="distance")
print(f"{'weight':>6}{'expansions':>12}{'distance':>10}")
for weight in [1, 1.5, 2, 3, 5]:
    result = measure(lambda p: astar_search(p, partial(weighted_heuristic, heuristic=straight_line, weight=weight)), long_route)
    print(f"{weight:>6}{result.expansions:>12}{result.solution.cost:>8.2f} km")

With `w > 1`, A\* expands far fewer nodes and gets more *greedy* (for very large `w`, it behaves like GBFS), though not steadily: here, `w = 3` expands more nodes than `w = 1.5`. The route is no longer guaranteed to be optimal: with `w = 1.5` it is already slightly longer (11.31 km instead of 11.29 km), `w = 2` happens to find the optimal one again, and larger weights return a route that is much longer than necessary. Over 100 routes:

In [ ]:
compare(routes, {
    'A* (straight-line)': lambda p: astar_search(p, straight_line),
    'A* (1.5 x straight-line)': lambda p: astar_search(p, partial(weighted_heuristic, heuristic=straight_line, weight=1.5)),
    'A* (2 x straight-line)': lambda p: astar_search(p, partial(weighted_heuristic, heuristic=straight_line, weight=2)),
    'A* (3 x straight-line)': lambda p: astar_search(p, partial(weighted_heuristic, heuristic=straight_line, weight=3)),
    'GBFS (straight-line)': lambda p: greedy_best_first_search(p, straight_line),
})

Overestimating trades solution quality for speed. Sometimes that is exactly what you want (a *good* route fast); but you lose the guarantee of an optimal one.

### Optional: Fastest routes

Can we do the same for the *fastest* route? Then the cost is in minutes. The straight-line distance in km is no longer a lower bound for the time, but we can divide it by the **highest speed** a vehicle ever reaches: nobody can get to the goal faster than flying straight there at top speed. For this part, we use the metro and tram network.

In [ ]:
def max_speed(transit_map: TransitMap) -> float:
    """
    Returns the highest speed in km per minute that any connection achieves,
    measured by the straight line between its stops.
    """
    return max(straight_line_km(transit_map.stops[origin], transit_map.stops[c.stop]) / c.minutes
               for origin, connections in transit_map.connections.items() for c in connections)

vienna_time = load_map('vienna', restrict=['metro', 'tram'])
TOP_SPEED = max_speed(vienna_time)  # in km per minute
print(f"top speed: {TOP_SPEED:.2f} km/min ({TOP_SPEED * 60:.1f} km/h)")

Implement the fastest possible time in minutes, using `TOP_SPEED` (in km per minute).

In [ ]:
def fastest_possible(state: JourneyState, problem: JourneyProblem) -> float:

    # TODO: return the time in minutes it would take to travel the straight-line
    # distance (in km) to the goal at the top speed (in km per minute)

    pass

In [ ]:
time_routes = random_routes(vienna_time, 'time')
time_landmarks = partial(landmark_heuristic, landmarks=choose_landmarks(vienna_time, 'time'), base=fastest_possible)
check_admissible(time_routes[0], fastest_possible)
compare(time_routes, {
    'Uniform Cost Search': uniform_cost_search,
    'A* (fastest possible)': lambda p: astar_search(p, fastest_possible),
    'A* (fastest possible + landmarks)': lambda p: astar_search(p, time_landmarks),
    'GBFS (fastest possible)': lambda p: greedy_best_first_search(p, fastest_possible),
})

The top speed (of the fastest metro section) is much higher than what most routes reach, so this heuristic is much looser than the one for distances, and saves less. The landmark heuristic, which knows real travel times, helps a lot more. (Skipping the optional parts above is fine: the next problem shows the same effects at a larger scale.)

## Second problem: Sliding puzzles

Vienna is small: UCS finds a route in milliseconds. To see why heuristics are *essential*, we need a problem with a huge state space. Let's look at **sliding-tile puzzles**: a board with numbered tiles and one empty space. An action slides a neighboring tile into the empty space (cost 1). The goal is to sort the tiles, with the empty space in the bottom right. The implementation lives in [puzzle.py](./puzzle.py).

In [ ]:
from puzzle import scrambled_puzzle, draw_puzzle, draw_puzzle_solution, PuzzleState, PuzzleProblem, EMPTY

problem = scrambled_puzzle(3, 12, seed=0)
draw_puzzle(problem.get_start())

`scrambled_puzzle(size, moves, seed)` makes random moves from the goal state, so it is solvable. Every move has a cost of 1, so the cost of a solution is its number of moves, and UCS finds a solution with the fewest moves. It can solve this easy one, which is only a few moves away from the goal; the highlighted tile is the one moved in each step:

In [ ]:
result = measure(uniform_cost_search, problem)
print(f"UCS: solution cost {result.solution.cost:.0f}, {result.expansions} expansions")
draw_puzzle_solution(result.solution)

The 3x3 puzzle ("8-puzzle") has 9!/2 = 181,440 reachable states, and some need up to 31 moves. Let us try a harder one that needs 26 moves (cost 26):

In [ ]:
hard = scrambled_puzzle(3, 60, seed=1)
result = measure(uniform_cost_search, hard)
print(f"UCS: solution cost {result.solution.cost:.0f}, {result.expansions} expansions, {result.seconds:.1f} seconds")

That took a while. Now the 4x4 puzzle ("15-puzzle"), which has 16!/2 ≈ 10,000,000,000,000 states. We let UCS give up after 200,000 expansions:

In [ ]:
hard15 = scrambled_puzzle(4, 60, seed=2)
result = measure(uniform_cost_search, hard15, budget=200_000)
print(f"UCS gave up: {result.gave_up}, after {result.expansions} expansions, {result.seconds:.1f} seconds")

UCS does not stand a chance. We need heuristics.

### Two admissible heuristics

We relax the puzzle to get lower bounds on the number of moves left:
- **Misplaced tiles:** Every tile that is not on its goal position needs at least one move. (Do not count the empty space!)
- **Manhattan distance:** Every tile needs at least as many moves as its distance to the goal position, measured in horizontal plus vertical steps (it cannot move diagonally, and only one tile moves per action).

Both are admissible. Which one is tighter? Since each misplaced tile is at least one step away, Manhattan distance is never smaller than the number of misplaced tiles: it dominates.

#### How is a puzzle state represented?

A state is a `PuzzleState` with two attributes:
- `state.size` is the width of the board (3 for the 8-puzzle, 4 for the 15-puzzle).
- `state.tiles` is a tuple of `size * size` numbers listing the tiles **row by row**, with `EMPTY` (= 0) marking the empty space. The tile in row `r` and column `c` (counted from 0) is `state.tiles[r * state.size + c]`.

`state.position(tile)` returns the `(row, column)` of a tile. The goal board is `problem.goal`, which is a `PuzzleState` as well. Here is an example:

In [ ]:
example = PuzzleState((1, 2, 3,
                       4, 5, 6,
                       0, 7, 8), 3)
print(example)                # printed as a board; the empty space is shown as "."
print(example.tiles)          # the tiles, row by row
print(example.position(7))    # tile 7 is in row 2, column 1
print(hard.goal.tiles)        # the goal board of a 3x3 puzzle

#### Misplaced tiles

Implement `misplaced_tiles()`: count the tiles that are not at their goal position. Compare the tiles of `state` with the tiles of `problem.goal` position by position, and do not count the empty space.

In [ ]:
def misplaced_tiles(state: PuzzleState, problem: PuzzleProblem) -> int:

    # TODO: count the tiles (not the empty space) that are not at their goal position

    pass

In [ ]:
assert misplaced_tiles(hard.goal, hard) == 0, "the goal should have no misplaced tiles"
assert misplaced_tiles(example, hard) == 2, "tiles 7 and 8 are misplaced, the empty space does not count"
swapped = PuzzleState((8, 2, 3, 4, 5, 6, 7, 1, 0), 3)
assert misplaced_tiles(swapped, hard) == 2, "tiles 1 and 8 are misplaced"
check_admissible(hard, misplaced_tiles)

#### Manhattan distance

Implement `manhattan()`: for each tile (not the empty space), take the horizontal plus vertical distance between its `(row, column)` in `state` and in `problem.goal`, and sum them up.

In [ ]:
def manhattan(state: PuzzleState, problem: PuzzleProblem) -> int:

    # TODO: sum up the horizontal plus vertical distances of all tiles (not the empty
    # space) between their position in `state` and in `problem.goal`

    pass

In [ ]:
assert manhattan(hard.goal, hard) == 0, "the goal should have zero distance"
assert manhattan(example, hard) == 2, "tiles 7 and 8 are each one step away from their goal position"
swapped = PuzzleState((8, 2, 3, 4, 5, 6, 7, 1, 0), 3)
assert manhattan(swapped, hard) == 6, "tiles 1 and 8 are each 3 steps (2 down/up, 1 left/right) away"
check_admissible(hard, manhattan)

Let's compare them on ten different 3x3 puzzles (this takes about 20 seconds, mostly because of UCS):

In [ ]:
puzzles = [scrambled_puzzle(3, 60, seed=seed) for seed in range(10)]
compare(puzzles, {
    'Uniform Cost Search': uniform_cost_search,
    'A* (misplaced tiles)': lambda p: astar_search(p, misplaced_tiles),
    'A* (Manhattan distance)': lambda p: astar_search(p, manhattan),
})

All three find optimal solutions. Each *tighter* admissible heuristic needs far fewer expansions. Now the 15-puzzle, where UCS gave up before:

In [ ]:
compare([hard15], {
    'Uniform Cost Search': uniform_cost_search,
    'A* (misplaced tiles)': lambda p: astar_search(p, misplaced_tiles),
    'A* (Manhattan distance)': lambda p: astar_search(p, manhattan),
}, budget=100_000)

Only the tighter heuristic solves it within the budget. A good heuristic does not make a search a bit faster: it can turn the *impossible* into the *possible*.

### Optional: Linear conflicts

Can you get even tighter? If two tiles are in their goal *row*, but in the wrong order, one of them has to leave the row and come back, which adds **two** moves that Manhattan distance does not count. The same goes for columns. For a group of conflicting tiles in a row, you need to remove as few tiles as possible to leave the rest in order; each removed tile adds 2 to the estimate. This **linear conflict** heuristic is Manhattan distance plus these extras, and still admissible.

In [ ]:
def conflicts(tiles_and_goals: list[int]) -> int:
    """
    Given the goal position of each tile in a line (listed in the order the tiles
    are in on the board), returns how many tiles must be removed from the line
    so that the rest is in the right order (n minus the longest increasing subsequence).
    """
    longest = [1] * len(tiles_and_goals)
    for i in range(len(tiles_and_goals)):
        for j in range(i):
            if tiles_and_goals[j] < tiles_and_goals[i]:
                longest[i] = max(longest[i], longest[j] + 1)
    return len(tiles_and_goals) - max(longest, default=0)

Use `conflicts()` to add 2 per conflicting tile to the Manhattan distance, for all rows and all columns. For each row, gather the tiles that are in that row and whose goal is also in that row, ordered by column. The values to pass to `conflicts()` are the goal columns of these tiles. Same for columns.

In [ ]:
def linear_conflict(state: PuzzleState, problem: PuzzleProblem) -> int:

    # TODO: start from manhattan(), then for each row and each column,
    # add 2 * conflicts(...) of the goal positions of the tiles that are in the right line

    pass

In [ ]:
check_admissible(hard, linear_conflict)
compare([hard15], {
    'A* (Manhattan distance)': lambda p: astar_search(p, manhattan),
    'A* (Manhattan + linear conflicts)': lambda p: astar_search(p, linear_conflict),
})

### Not admissible: weighted Manhattan distance

Again we can trade optimality for speed by overestimating. How much do we lose on the sliding puzzles?

In [ ]:
compare(puzzles, {
    'A* (Manhattan distance)': lambda p: astar_search(p, manhattan),
    'A* (1.5 x Manhattan)': lambda p: astar_search(p, partial(weighted_heuristic, heuristic=manhattan, weight=1.5)),
    'A* (2 x Manhattan)': lambda p: astar_search(p, partial(weighted_heuristic, heuristic=manhattan, weight=2)),
    'A* (3 x Manhattan)': lambda p: astar_search(p, partial(weighted_heuristic, heuristic=manhattan, weight=3)),
    'GBFS (Manhattan distance)': lambda p: greedy_best_first_search(p, manhattan),
})

With a small weight, weighted A\* gets solutions with much fewer expansions, and they are only a little longer than optimal. With large weights and in GBFS, they get much longer: we are shuffling tiles around for no good reason. (Note that more weight does not always mean fewer expansions: once the heuristic is far off, the search can wander.)

## Summary

- **A\*** expands nodes by `g(n) + h(n)`; with `h = 0` it is UCS, without `g` it is GBFS.
- With an **admissible** (never overestimates) and **consistent** (`h(s) ≤ cost(a) + h(s')` for every action `a` from `s` to `s'`) heuristic, A\* with a closed set, as in our implementation, finds an optimal solution. GBFS does not.
- Of two admissible heuristics, the **larger** one dominates, and needs fewer expansions. On large state spaces, this makes the difference between solved and unsolved.
- An **inadmissible** heuristic may speed up the search, but forfeits the optimality guarantee.

### Questions to think about
1. Why is the straight-line distance admissible for distance costs? Why would it not be, if connections could have a shorter distance than the straight line?
2. Is `h(state) = 0` for the goal state and `1` for all other states admissible for the sliding puzzle? Is it admissible for journey planning by distance in km? What would A\* do with the *perfect* heuristic, the true remaining cost?
3. We used the *Manhattan distance* for puzzle tiles. Would the *Euclidean distance* or the *Chebyshev distance* of each tile be admissible? Would they be tighter or looser?
4. The weighted heuristic made GBFS-like behavior with `w → ∞`. What does it do for `w → 0`?